# Tutorial 2: The Data Pipeline

## From Raw Simulations to Batched Tensors

---

This notebook covers the full data pipeline — every step between calling `simulate_pulsar_factorized` and handing a batch tensor to the model. The pipeline must handle **variable-length, irregularly sampled** sequences from multiple backends, which is a central challenge for PTA data.

### What you'll learn

1. How raw TOA data are converted to 6 continuous token features (tokenization)
2. The signed-log transform that tames extreme signal-to-noise ratios
3. Four structured masking augmentations that simulate realistic data degradation
4. On-the-fly (`PulsarDataset`) vs pre-generated (`FixedPulsarDataset`) datasets in factorized mode
5. The collation step: padding, mask construction, and the `theta_global`/`theta_wn` split

### Modules covered

| Module | Purpose |
|--------|---------|
| `src.models.tokenization` | Feature engineering per TOA |
| `src.masking` | Data augmentation via structured removal |
| `src.dataset` | PyTorch datasets with on-the-fly simulation |
| `src.collate` | Padding and batch construction |

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED = 42

---
## 1. Tokenization

Each TOA in a pulsar dataset is transformed into a **token** — a fixed-dimensional feature vector that the model can process. The `tokenize` function computes 6 continuous features + 1 categorical feature per observation:

| # | Feature | Formula | Rationale |
|---|---------|---------|----------|
| 0 | `t_norm` | $(t_i - t_{\min}) / T_{\rm span}$ | Normalised position in $[0, 1]$ |
| 1 | `dt_prev` | $\Delta t_i / T_{\rm span}$ | Gap since previous TOA (cadence info) |
| 2 | `r_over_sig` | $\text{sign}(r/\sigma) \cdot \log(1 + |r/\sigma|)$ | Scaled residual with signed-log compression |
| 3 | `log_sigma` | $\log_{10}(\sigma_i)$ | Noise level per observation |
| 4 | `r_raw` | $r_i$ (in seconds) | Unscaled residual |
| 5 | `freq_norm` | $(f_{\rm MHz} - 1400) / 1000$ | Observing frequency centred on L-band |
| — | `backend_id` | integer $\in \{0, 1, 2, \ldots\}$ | Receiver system (embedded separately) |

In [ ]:
from src.schedules import generate_schedule
from src.simulator import simulate_pulsar
from src.models.tokenization import tokenize, N_CONTINUOUS_FEATURES

# Simulate one pulsar
rng = np.random.default_rng(SEED)
theta = np.array([-13.5, 4.0])
sched = generate_schedule(rng)
sim = simulate_pulsar(theta, sched, n_modes=30, rng=rng)

# Tokenize
tokens = tokenize(sim.t, sim.sigma, sim.residuals, sim.freq_mhz, sim.backend_id)

print(f'N_CONTINUOUS_FEATURES = {N_CONTINUOUS_FEATURES}')
print(f'Number of TOAs: {len(sim.t)}')
print(f'Token keys: {list(tokens.keys())}')
for k, v in tokens.items():
    print(f'  {k:12s}  shape={str(v.shape):10s}  dtype={v.dtype}  '
          f'range=[{v.min().item():.4f}, {v.max().item():.4f}]')

In [ ]:
# Visualise all 6 continuous features for this pulsar
feat_names = ['t_norm', 'dt_prev', 'r_over_sig', 'log_sigma', 'r_raw', 'freq_norm']
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)

t_norm = tokens['t_norm'].numpy()
for ax, key in zip(axes.ravel(), feat_names):
    vals = tokens[key].numpy()
    ax.plot(t_norm, vals, '.', ms=3, alpha=0.5)
    ax.set_ylabel(key)
    ax.set_title(f'{key}  (μ={vals.mean():.3f}, σ={vals.std():.3f})', fontsize=9)

for ax in axes[-1]:
    ax.set_xlabel('t_norm')

fig.suptitle(f'Token features — $\\log_{{10}}A={theta[0]}$, $\\gamma={theta[1]}$, '
             f'N={len(sim.t)}', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

### The signed-log transform

The key design choice is the **signed-log transform** on `r_over_sig`:

$$
\texttt{r\_over\_sig} = \text{sign}(r_i / \sigma_i) \cdot \log(1 + |r_i / \sigma_i|)
$$

Without this, the raw $r/\sigma$ can reach $10^5$ for strong red noise signals, which would:
- Overflow in float16 during mixed-precision training
- Dominate the feature vector, washing out other channels

In [ ]:
# Demonstrate the signed-log transform
x = np.linspace(-200, 200, 1000)
y = np.sign(x) * np.log1p(np.abs(x))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, y, lw=1.5)
axes[0].set_xlabel('Raw r/σ'); axes[0].set_ylabel('Transformed')
axes[0].set_title('Signed-log transform: sign(x)·log(1+|x|)')
axes[0].axhline(0, color='gray', ls='--', lw=0.5)
axes[0].axvline(0, color='gray', ls='--', lw=0.5)

# Show distribution before/after on actual data
raw_ros = sim.residuals / (sim.sigma + 1e-10)
transformed = np.sign(raw_ros) * np.log1p(np.abs(raw_ros))

axes[1].hist(raw_ros, bins=50, alpha=0.6, label=f'Raw r/σ (range [{raw_ros.min():.0f}, {raw_ros.max():.0f}])',
             density=True)
axes[1].hist(transformed, bins=50, alpha=0.6, label=f'Signed-log (range [{transformed.min():.1f}, {transformed.max():.1f}])',
             density=True)
axes[1].legend(fontsize=8)
axes[1].set_xlabel('Value'); axes[1].set_ylabel('Density')
axes[1].set_title('Distribution compression on real data')

fig.tight_layout()
plt.show()

---
## 2. Masking Augmentations

Real PTA datasets suffer from **missing data**: telescopes go offline, campaigns are interrupted, or data are flagged and removed. To make models robust, we augment training data by deliberately removing observations in structured patterns.

Four augmentation types are implemented:

| Augmentation | Description | Real-world analogue |
|-------------|-------------|--------------------|
| **Season dropout** | Drop all TOAs within 1+ random ~8-month windows | Telescope downtime |
| **End truncation** | Keep only the first 40–85% of the timeline | Shorter baseline |
| **Cadence thinning** | Randomly drop individual TOAs | Irregular cadence |
| **Compound** | Season dropout + thinning | Multiple issues |

In [ ]:
from src.masking import season_dropout, end_truncation, cadence_thinning, apply_random_masking

# Generate a long, well-sampled schedule for clear visualization
rng_m = np.random.default_rng(1234)
sched_m = generate_schedule(rng_m, tspan_min_yr=12, tspan_max_yr=12,
                             n_toa_min=300, n_toa_max=300)
t = sched_m.t

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

# Original data
axes[0].scatter(t, np.ones_like(t), s=8, alpha=0.6, c='black')
axes[0].set_title(f'Original: N={len(t)} TOAs', fontsize=10)
axes[0].set_yticks([])

# Season dropout
rng_sd = np.random.default_rng(42)
keep_sd = season_dropout(t, rng_sd, n_drop=2)
axes[1].scatter(t[keep_sd], np.ones(keep_sd.sum()), s=8, alpha=0.6, c='C0')
axes[1].scatter(t[~keep_sd], np.ones((~keep_sd).sum()), s=8, alpha=0.3, c='red', marker='x')
axes[1].set_title(f'Season dropout (2 seasons): kept {keep_sd.sum()}/{len(t)}', fontsize=10)
axes[1].set_yticks([])

# End truncation
rng_et = np.random.default_rng(42)
keep_et = end_truncation(t, rng_et, min_frac=0.4, max_frac=0.6)
axes[2].scatter(t[keep_et], np.ones(keep_et.sum()), s=8, alpha=0.6, c='C1')
axes[2].scatter(t[~keep_et], np.ones((~keep_et).sum()), s=8, alpha=0.3, c='red', marker='x')
axes[2].set_title(f'End truncation: kept {keep_et.sum()}/{len(t)}', fontsize=10)
axes[2].set_yticks([])

# Cadence thinning
rng_ct = np.random.default_rng(42)
keep_ct = cadence_thinning(t, rng_ct, keep_prob=0.4)
axes[3].scatter(t[keep_ct], np.ones(keep_ct.sum()), s=8, alpha=0.6, c='C2')
axes[3].scatter(t[~keep_ct], np.ones((~keep_ct).sum()), s=8, alpha=0.3, c='red', marker='x')
axes[3].set_title(f'Cadence thinning (p=0.4): kept {keep_ct.sum()}/{len(t)}', fontsize=10)
axes[3].set_yticks([])

# Compound
rng_cmp = np.random.default_rng(42)
keep_cmp = season_dropout(t, rng_cmp, n_drop=1)
keep_cmp &= cadence_thinning(t, rng_cmp, keep_prob=0.6)
axes[4].scatter(t[keep_cmp], np.ones(keep_cmp.sum()), s=8, alpha=0.6, c='C4')
axes[4].scatter(t[~keep_cmp], np.ones((~keep_cmp).sum()), s=8, alpha=0.3, c='red', marker='x')
axes[4].set_title(f'Compound (season + thinning): kept {keep_cmp.sum()}/{len(t)}', fontsize=10)
axes[4].set_yticks([])
axes[4].set_xlabel('Time (yr)')

fig.suptitle('Masking Augmentation Strategies', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# How apply_random_masking works under different severity levels
severities = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
n_trials = 200
keep_fractions = {s: [] for s in severities}

for s in severities:
    for trial in range(n_trials):
        rng_trial = np.random.default_rng(5000 + trial)
        keep = apply_random_masking(t, rng_trial, severity=s)
        keep_fractions[s].append(keep.sum() / len(t))

fig, ax = plt.subplots(figsize=(8, 4))
positions = range(len(severities))
bp = ax.boxplot([keep_fractions[s] for s in severities], positions=positions,
                widths=0.5, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
ax.set_xticks(positions)
ax.set_xticklabels([f'{s:.1f}' for s in severities])
ax.set_xlabel('Masking severity')
ax.set_ylabel('Fraction of TOAs kept')
ax.set_title(f'Data retention vs severity ({n_trials} trials per level, N={len(t)} TOAs)')
ax.axhline(20/len(t), color='red', ls='--', lw=1, label=f'min_remaining={20}')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

---
## 3. Datasets: On-the-Fly vs Pre-Generated

Two dataset classes serve different purposes:

| Class | Use case | θ source | Masking | Stores sim? |
|-------|----------|----------|---------|-------------|
| `PulsarDataset` | Training & validation | `FactorizedPrior` (IID or quasi-random bank) | On-the-fly | No |
| `FixedPulsarDataset` | Evaluation | `FactorizedPrior` IID sample | None | Yes (for exact posterior) |

### PulsarDataset (training)

`PulsarDataset.__getitem__(idx)` performs the full pipeline deterministically from `seed + idx` (or from `seed + idx + epoch × n_samples` when `reseed_per_epoch=True`):
1. Draw `theta_global` and `theta_wn` from `FactorizedPrior`
2. Generate a fresh schedule with 1–4 backends
3. Simulate red + DM noise + per-backend EFAC/EQUAD/ECORR white noise
4. Apply random masking (with severity sampled from `[0, masking_severity]`)
5. Tokenize the surviving TOAs

In [ ]:
from src.priors import FactorizedPrior
from src.dataset import PulsarDataset, FixedPulsarDataset

global_bounds = {
    'log10_A_red': [-18, -11], 'gamma_red': [1.0, 7.0],
    'log10_A_dm':  [-18, -11], 'gamma_dm':  [1.0, 7.0],
}
wn_bounds = {'EFAC': [0.5, 2.0], 'log10_EQUAD': [-8, -5], 'log10_ECORR': [-8, -5]}
prior = FactorizedPrior(global_bounds, wn_bounds, n_backends_max=4)

data_cfg = {'n_fourier_modes': 30, 'n_toa_min': 80, 'n_toa_max': 400,
            'tspan_min_yr': 5.0, 'tspan_max_yr': 15.0}

# Training dataset — factorized=True returns theta_global and theta_wn
train_ds = PulsarDataset(
    n_samples=1000,
    prior=prior,
    data_cfg=data_cfg,
    seed=42,
    masking_severity=0.5,
    augment=True,
    use_sobol=False,
    factorized=True,
)

# Fetch a single item
item = train_ds[0]
print('Keys:', list(item.keys()))
print(f'  theta_global: {item["theta_global"]}  (log10_A_red, gamma_red, log10_A_dm, gamma_dm)')
print(f'  theta_wn:     shape={item["theta_wn"].shape}  (n_backends_max, 3) — EFAC, EQUAD, ECORR')
print(f'  seq_len:      {item["seq_len"]} TOAs after masking')
print(f'  tspan_yr:     {item["tspan_yr"]:.2f} years')

In [ ]:
# Demonstrate deterministic reproducibility
item_a = train_ds[17]
item_b = train_ds[17]
print('Same index, same result?',
      torch.allclose(item_a['tokens']['r_over_sig'], item_b['tokens']['r_over_sig']))

# Show sequence length variation across the dataset
seq_lens = [train_ds[i]['seq_len'] for i in range(200)]
print(f'\nSequence lengths (first 200): {np.mean(seq_lens):.0f} ± {np.std(seq_lens):.0f} '
      f'(range {min(seq_lens)}–{max(seq_lens)})')

In [ ]:
# FixedPulsarDataset: stores full sim for exact posterior evaluation
eval_ds = FixedPulsarDataset(
    n_samples=5,
    prior=prior,
    data_cfg=data_cfg,
    seed=100000,
    factorized=True,
)

item_e = eval_ds[0]
print('FixedPulsarDataset item keys:', list(item_e.keys()))
print(f'  Has sim object: {"sim" in item_e}')
print(f'  theta_global:   {item_e["theta_global"].numpy()}')
print(f'  theta_wn:       shape={item_e["theta_wn"].shape}  (n_backends_max, 3)')
print(f'  sim.F shape: {item_e["sim"].F.shape}   (Fourier design matrix)')
print(f'  sim.tspan: {item_e["sim"].tspan:.2f} yr')
print('\nThe sim object is needed to compute the exact Bayesian posterior')
print('for quantitative model evaluation (Hellinger distance, etc.).')

---
## 4. Collation: Building Padded Batches

Since sequences have different lengths, we can't naively stack them. The `collate_fn` function:

1. Finds the maximum sequence length $L_{\max}$ in the batch
2. Pads all sequences to $L_{\max}$ (zeros for features, False for mask)
3. Stacks the 6 continuous features into a `(B, L_{max}, 6)` tensor
4. Creates a boolean `mask` of shape `(B, L_{max})` where `True` = real token

The feature ordering is: `[t_norm, dt_prev, r_over_sig, log_sigma, r_raw, freq_norm]`

In [ ]:
from src.collate import collate_fn

# Manually collate a small batch
batch_items = [train_ds[i] for i in range(4)]
batch = collate_fn(batch_items)

print('Batch contents:')
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:12s}  shape={str(list(v.shape)):20s}  dtype={v.dtype}')

print(f'\nSequence lengths: {batch["seq_lens"].tolist()}')
print(f'Padded length: {batch["features"].shape[1]}')

# Verify mask correctness
for i in range(4):
    L = batch['seq_lens'][i].item()
    assert batch['mask'][i, :L].all(), f'First {L} should be True'
    assert not batch['mask'][i, L:].any(), f'After {L} should be False'
print('\nMask integrity check passed ✓')

In [ ]:
# Visualise the padded batch structure
fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Feature heatmap
im = axes[0].imshow(batch['features'][0].numpy().T, aspect='auto', cmap='RdBu_r',
                     vmin=-3, vmax=3)
axes[0].set_yticks(range(6))
axes[0].set_yticklabels(['t_norm', 'dt_prev', 'r/σ', 'log_σ', 'r_raw', 'freq'])
axes[0].set_xlabel('Token position')
axes[0].set_title('Feature tensor — sample 0 (padded)', fontsize=10)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Mask heatmap for all 4 samples
axes[1].imshow(batch['mask'].numpy(), aspect='auto', cmap='Blues',
               interpolation='nearest')
axes[1].set_xlabel('Token position')
axes[1].set_ylabel('Batch index')
axes[1].set_title('Attention mask (blue = valid, white = padding)', fontsize=10)
for i in range(4):
    L = batch['seq_lens'][i].item()
    axes[1].axvline(L - 0.5, color='red', lw=0.8, alpha=0.7)

fig.tight_layout()
plt.show()

---
## 5. Full DataLoader Pipeline

Putting it all together with a PyTorch DataLoader:

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)

# Iterate one batch
batch = next(iter(loader))
print('Full DataLoader batch:')
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:14s}  shape={str(list(v.shape)):25s}  dtype={v.dtype}')

print(f'\nFactorized batch has separate theta tensors:')
print(f'  theta_global → (B, 4): 4D global params for the global flow loss')
print(f'  theta_wn     → (B, 4, 3): per-backend WN params for the WN flow loss')
print(f'Sequence lengths: min={batch["seq_lens"].min()}, max={batch["seq_lens"].max()}')

---
## Summary

The data pipeline transforms raw pulsar simulations into model-ready tensor batches:

```
schedule, θ_global, θ_wn  →  simulate_pulsar_factorized  →  apply_random_masking  →  tokenize  →  collate_fn  →  batch dict
```

### Key takeaways

1. **Tokenization** extracts 6 informative features per TOA, with a signed-log transform that prevents float16 overflow
2. **Masking augmentations** simulate 4 types of realistic data loss, controlled by a severity parameter
3. **PulsarDataset** with `factorized=True` draws `theta_global` (4D) and `theta_wn` (3D per backend) from `FactorizedPrior`
4. **`reseed_per_epoch=True`** regenerates all (θ, x) pairs each epoch, preventing the flow from memorising fixed training pairs
5. **FixedPulsarDataset** pre-generates and stores full simulations for evaluation with exact posteriors
6. **collate_fn** pads variable-length sequences and separates `theta_global` / `theta_wn` for the factorized loss

**Next**: [Tutorial 3 — Model Architecture](03_model_architecture.ipynb) covers the transformer encoder, factorized flows, and how they combine.